## 3自由度坐标变换

In [ ]:
def transf_grillage(theta, n_nodes=2, degree=True):
    """
    生成二维 frame / beam 单元的坐标变换矩阵

    每个节点 3 个自由度:
        [v, rz, rx]

    对于 n_nodes 个节点，总自由度数为:
        3 * n_nodes

    Parameters
    ----------
    theta : float
        单元局部 x 轴相对于全局 x 轴的夹角

    n_nodes : int
        节点数，二节点 frame element 取 n_nodes=2

    degree : bool
        True  表示 theta 使用角度制
        False 表示 theta 使用弧度制
    """

    if degree:
        theta = np.deg2rad(theta)

    c = np.cos(theta)
    s = np.sin(theta)

    T_node = np.array([
        [ 1,  0, 0],
        [ 0,  c, s],
        [ 0, -s, c]
    ], dtype=float)

    T = np.kron(np.eye(n_nodes), T_node)

    return T

## 单元刚度矩阵

In [ ]:
def stiffness_grillage(theta, EI, GJ, L, degree=True):
    t = transf_grillage(theta, degree=degree)
    t_T = t.T

    k = np.array([
        [ 12*EI/L**3,   6*EI/L**2,       0, -12*EI/L**3,   6*EI/L**2,       0],
        [  6*EI/L**2,     4*EI/L,        0,  -6*EI/L**2,     2*EI/L,        0],
        [           0,          0,     GJ/L,           0,          0,    -GJ/L],
        [-12*EI/L**3,  -6*EI/L**2,       0,  12*EI/L**3,  -6*EI/L**2,       0],
        [  6*EI/L**2,     2*EI/L,        0,  -6*EI/L**2,     4*EI/L,        0],
        [           0,          0,    -GJ/L,           0,          0,     GJ/L]
    ], dtype=float)

    k = t_T @ k @ t

    return k


## 组装

In [ ]:
def assemble_K_grillage(ke_list, connectivity, n_nodes, numbering="one"):
    """
    组装 grillage element 的总体刚度矩阵

    每个节点 3 个自由度:
        [v, theta_z, theta_x]

    每个单元刚度矩阵自由度顺序:
        [vi, theta_zi, theta_xi, vj, theta_zj, theta_xj]

    Parameters
    ----------
    ke_list : list
        单元全局刚度矩阵列表，例如 [k1, k2, k3]
        每个 ke 应为 6x6 矩阵

    connectivity : list
        单元连接关系，例如:
        [(1, 2), (2, 3)]

    n_nodes : int
        总节点数

    numbering : str
        "one"  表示节点编号从 1 开始
        "zero" 表示节点编号从 0 开始

    Returns
    -------
    K : ndarray
        总体刚度矩阵
    """

    if len(ke_list) != len(connectivity):
        raise ValueError("ke_list and connectivity must have the same length.")

    dof_per_node = 3
    total_dof = dof_per_node * n_nodes

    K = np.zeros((total_dof, total_dof))

    for ke, conn in zip(ke_list, connectivity):

        ke = np.asarray(ke, dtype=float)

        if ke.shape != (6, 6):
            raise ValueError("Each grillage element stiffness matrix must be 6x6.")

        ni, nj = conn

        if numbering == "one":
            ni -= 1
            nj -= 1
        elif numbering == "zero":
            pass
        else:
            raise ValueError("numbering must be 'one' or 'zero'.")

        dofs = [
            3 * ni,      # v_i
            3 * ni + 1,  # theta_zi
            3 * ni + 2,  # theta_xi
            3 * nj,      # v_j
            3 * nj + 1,  # theta_zj
            3 * nj + 2   # theta_xj
        ]

        K[np.ix_(dofs, dofs)] += ke

    return K


## 自由度定义

In [ ]:
def dof_grillage(node, direction, numbering="one"):
    """
    grillage element 的自由度编号函数

    每个节点 3 个自由度:
        v        : 竖向位移
        theta_z  : 弯曲转角
        theta_x  : 扭转转角

    Parameters
    ----------
    node : int
        节点编号

    direction : str
        自由度方向，可输入:
        "v", "y"                         -> 竖向位移
        "theta_z", "thetaz", "rz", "z"    -> 绕 z 轴转角
        "theta_x", "thetax", "rx", "x"    -> 绕 x 轴转角

    numbering : str
        "one"  表示节点编号从 1 开始
        "zero" 表示节点编号从 0 开始

    Returns
    -------
    index : int
        对应的总体自由度编号
    """

    if numbering == "one":
        node -= 1
    elif numbering == "zero":
        pass
    else:
        raise ValueError("numbering must be 'one' or 'zero'.")

    direction = direction.lower()

    if direction in ["v", "y"]:
        return 3 * node

    elif direction in ["theta_z", "thetaz", "rz", "z"]:
        return 3 * node + 1

    elif direction in ["theta_x", "thetax", "rx", "x"]:
        return 3 * node + 2

    else:
        raise ValueError("direction must be 'v', 'theta_z', or 'theta_x'.")


## 求解函数

In [1]:
def solve_structure(K, force_bc=None, disp_bc=None):
    """
    求解总体刚度方程 K d = F

    Parameters
    ----------
    K : ndarray
        总体刚度矩阵

    force_bc : dict
        已知外力边界条件，格式为：
        {
            dof_index: force_value
        }

        例如：
        {
            3: -50
        }

    disp_bc : dict
        已知位移边界条件，格式为：
        {
            dof_index: displacement_value
        }

        例如：
        {
            0: 0,
            1: 0,
            4: 0,
            5: 0
        }

    Returns
    -------
    d : ndarray
        完整位移向量

    reaction : ndarray
        完整反力向量

    free_dofs : ndarray
        自由自由度编号

    fixed_dofs : ndarray
        约束自由度编号
    """

    K = np.asarray(K, dtype=float)
    n_dof = K.shape[0]

    if K.shape[0] != K.shape[1]:
        raise ValueError("K must be a square matrix.")

    # -------------------------
    # 1. 定义总外力向量 F
    # -------------------------
    F = np.zeros(n_dof)

    if force_bc is not None:
        for index, value in force_bc.items():
            F[index] = value

    # -------------------------
    # 2. 定义已知位移向量 d
    # -------------------------
    d = np.zeros(n_dof)

    if disp_bc is None:
        disp_bc = {}

    fixed_dofs = np.array(sorted(disp_bc.keys()), dtype=int)

    for index, value in disp_bc.items():
        d[index] = value

    # -------------------------
    # 3. 找自由自由度
    # -------------------------
    all_dofs = np.arange(n_dof)
    free_dofs = np.setdiff1d(all_dofs, fixed_dofs)

    # -------------------------
    # 4. 分块矩阵
    # -------------------------
    K_ff = K[np.ix_(free_dofs, free_dofs)]
    K_fc = K[np.ix_(free_dofs, fixed_dofs)]

    F_f = F[free_dofs]
    d_c = d[fixed_dofs]

    # -------------------------
    # 5. 求解自由位移
    # -------------------------
    rhs = F_f - K_fc @ d_c
    d_f = np.linalg.solve(K_ff, rhs)

    d[free_dofs] = d_f

    # -------------------------
    # 6. 计算支座反力
    # -------------------------
    internal_force = K @ d
    reaction = internal_force - F

    return d, reaction, free_dofs, fixed_dofs